# 11 — Capstone Project: The Lumina Support Assistant

Time to put the whole course together. We'll build a **complete support assistant** for the
fictional Lumina Robotics — the kind of thing you could actually deploy behind a chat
widget. It combines almost everything you've learned:

| Capability | Notebook it came from |
|------------|----------------------|
| Chat model + prompts | 02, 03 |
| Structured output (a typed support ticket) | 04 |
| Document loading, splitting, embeddings, vector store | 07, 08 |
| Retrieval-augmented answers with sources | 09 |
| Tools the model can call, an agent that decides | 10 |
| Conversation memory across turns | 06, 10 |

We'll package it as a clean Python **class** — `LuminaSupportAssistant` — so it's reusable
and easy to extend, exactly how you'd structure it in a real codebase.

**What the assistant can do:**
1. Answer product questions from the Lumina knowledge base (RAG via a tool).
2. Create a structured **support ticket** when a customer reports a problem.
3. Remember the conversation so follow-ups make sense.
4. Decline politely when something is outside its knowledge.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."
assert os.path.exists("data/lumina_handbook.md"), "Run `python make_data.py` first."
print("Environment ready.")

## 1. The support ticket schema

When a customer has a real problem, the assistant should capture it as structured data the
business can act on — not just chat. We define the ticket shape with Pydantic and constrain
`category` and `priority` to fixed values (notebook 04).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class SupportTicket(BaseModel):
    '''A structured support ticket created from a customer's problem.'''
    subject: str = Field(description="A short one-line summary of the issue")
    category: Literal["hardware", "software", "billing", "other"] = Field(
        description="The type of problem"
    )
    priority: Literal["low", "medium", "high"] = Field(
        description="Urgency: high if the device is unusable or the customer is very upset"
    )
    details: str = Field(description="A clear description of the problem in 1-3 sentences")

print("Ticket schema defined.")

## 2. The assistant class

Here's the full assistant. Read it top to bottom — each private method maps to a concept
from the course:

- `_build_retriever` — load, split, embed, store (notebooks 07–08).
- `_build_tools` — two tools: one searches the docs (agentic RAG), one files a ticket. They
  are defined as inner functions so they can access `self.retriever` and `self.tickets`.
- `_build_agent` — `create_agent` with a system prompt and an in-memory checkpointer for
  memory (notebook 10).
- `chat` — the public method you call each turn, keyed by a `thread_id`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


SYSTEM_PROMPT = (
    "You are the customer support assistant for Lumina Robotics, maker of the Pixel home "
    "robot. Follow these rules:\n"
    "1. For any question about Lumina or Pixel (price, battery, warranty, returns, "
    "   features, support), call `search_docs` and answer ONLY from what it returns. If the "
    "   docs don't cover it, say you don't know and offer to file a ticket.\n"
    "2. When the customer reports a malfunction or a problem that needs human follow-up, "
    "   call `file_ticket` to record it, then tell the customer their ticket was created.\n"
    "3. Be concise, friendly, and never invent product facts."
)


class LuminaSupportAssistant:
    def __init__(self, model_name: str = "gemini-2.5-flash", data_dir: str = "data"):
        self.model = ChatGoogleGenerativeAI(model=model_name, temperature=0)
        self.embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
        self.tickets: list[SupportTicket] = []          # our "database" of filed tickets
        self.retriever = self._build_retriever(data_dir)
        self.agent = self._build_agent()

    def _build_retriever(self, data_dir: str):
        docs = DirectoryLoader(
            data_dir, glob="**/*.md", loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        ).load()
        chunks = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=80
        ).split_documents(docs)
        store = InMemoryVectorStore.from_documents(chunks, embedding=self.embeddings)
        return store.as_retriever(search_kwargs={"k": 3})

    def _build_tools(self):
        retriever = self.retriever
        tickets = self.tickets

        @tool
        def search_docs(query: str) -> str:
            '''Search the Lumina Robotics knowledge base for product facts: pricing,
            battery, warranty, returns, languages, specs, support hours.'''
            found = retriever.invoke(query)
            return "\n\n".join(d.page_content for d in found)

        @tool
        def file_ticket(subject: str,
                        category: Literal["hardware", "software", "billing", "other"],
                        priority: Literal["low", "medium", "high"],
                        details: str) -> str:
            '''File a support ticket for a customer problem that needs human follow-up.
            Use this for malfunctions, defects, billing disputes, or anything the docs
            cannot resolve.'''
            ticket = SupportTicket(subject=subject, category=category,
                                   priority=priority, details=details)
            tickets.append(ticket)
            return f"Ticket #{len(tickets)} filed: [{ticket.priority}] {ticket.subject}"

        return [search_docs, file_ticket]

    def _build_agent(self):
        return create_agent(
            self.model,
            tools=self._build_tools(),
            system_prompt=SYSTEM_PROMPT,
            checkpointer=InMemorySaver(),
        )

    def chat(self, message: str, thread_id: str = "default") -> str:
        config = {"configurable": {"thread_id": thread_id}}
        result = self.agent.invoke({"messages": [{"role": "user", "content": message}]},
                                   config=config)
        return result["messages"][-1].content


assistant = LuminaSupportAssistant()
print("Lumina Support Assistant is online.")

## 3. A realistic conversation

Let's run a multi-turn support session on a single `thread_id` so the assistant remembers
context. Watch how it uses different capabilities for different turns.

In [ ]:
SESSION = "customer-42"

def turn(message):
    print("Customer:", message)
    print("Assistant:", assistant.chat(message, thread_id=SESSION), "\n")

# Turn 1 - a product question -> should search the docs (RAG)
turn("Hi! How long is the warranty on the Pixel?")

# Turn 2 - a follow-up that only makes sense with memory
turn("And can I return it if I change my mind?")

In [ ]:
# Turn 3 - small talk that needs no tool
turn("Great, thanks. You've been helpful!")

# Turn 4 - a malfunction report -> should file a ticket
turn("Actually, one problem: my Pixel Pro keeps shutting off randomly after the last update.")

## 4. Inspect what the assistant did

Because tickets are stored on the object, we can confirm the assistant didn't just *say* it
filed one — it actually created structured data we can use downstream (route to a team,
store in a database, page an on-call engineer for high priority).

In [ ]:
print(f"{len(assistant.tickets)} ticket(s) on file:\n")
for i, t in enumerate(assistant.tickets, 1):
    print(f"Ticket #{i}")
    print("  subject :", t.subject)
    print("  category:", t.category)
    print("  priority:", t.priority)
    print("  details :", t.details)
    print()

Notice the ticket is a validated `SupportTicket` object: `category` and `priority` are
guaranteed to be from the allowed sets, so the rest of your system can trust them. That's
the payoff of combining an agent (decides *when* to file) with structured output (decides
*what shape* the data takes).

## 5. Confirming the boundaries

A trustworthy assistant knows its limits. Ask about something outside the knowledge base
and it should decline rather than fabricate — and, per its instructions, offer to file a
ticket instead.

In [ ]:
print(assistant.chat(
    "Does the Pixel integrate with my Tesla and unlock my car?",
    thread_id="customer-99",
))

## 6. Where to take it next

This capstone is deliberately compact so the whole flow fits in your head. To make it
production-ready you would:

- **Persist state.** Swap `InMemoryVectorStore` for Chroma/pgvector and `InMemorySaver` for
  a database checkpointer, and write tickets to a real datastore.
- **Add tools.** Order lookup, refund initiation, scheduling a callback — each a new `@tool`.
- **Stream responses.** Use the agent's streaming interface for a responsive chat UI.
- **Guard and observe.** Add input validation, rate limits, and tracing (LangSmith) to see
  every tool call and prompt in production.
- **Evaluate.** Build a small test set of questions with expected answers and check the
  assistant against it whenever you change a prompt or model.

The architecture, though, stays exactly what you built here: **retrieve for knowledge,
tools for actions, an agent to orchestrate, memory for continuity, and structured output
where the data must be trusted.**

## Your turn

Five exercises that extend the assistant the way a production project would — new tools (support
hours, order status), isolated per-customer memory, escalating urgent tickets, and a smoke-test
to catch regressions. Try each in a blank cell before opening its solution.

**Exercise 1 — Support-hours tool.** Add a `check_support_hours()` tool so the assistant can
answer "are you open right now?", wire it into the system prompt, rebuild, and test it.

*Sample run (illustrative — the rebuilt assistant answering the test question):*

```text
We're open Monday to Friday, 9am to 6pm Central Standard Time, so yes — we're open right now!
```

<details><summary>Show solution</summary>

Add inside `_build_tools` (and include it in the returned list):

```python
@tool
def check_support_hours() -> str:
    '''Return Lumina Robotics support hours.'''
    return "Support is open Monday-Friday, 9am-6pm Central Standard Time."
```

Then mention it in `SYSTEM_PROMPT` ("For questions about availability/hours, call
`check_support_hours`."), rebuild with `assistant = LuminaSupportAssistant()`, and try:

```python
print(assistant.chat("Are you open right now?", thread_id="hours-test"))
```

</details>

**Exercise 2 — Order-status tool.** Add a `check_order_status(order_id)` tool backed by a small
dict so the assistant can answer "where's my order 1234?" — a brand-new capability with no
other code changes.

*Sample run (illustrative):*

```text
Good news — your order 1234 has shipped and is arriving Friday.
```

<details><summary>Show solution</summary>

Add inside `_build_tools` and return it alongside the others:

```python
@tool
def check_order_status(order_id: str) -> str:
    '''Look up the delivery status of an order by its ID, e.g. "1234".'''
    orders = {"1234": "shipped, arriving Friday", "5678": "still processing"}
    return orders.get(order_id, "no order found with that ID")
```

Mention it in `SYSTEM_PROMPT` ("For order/delivery questions, call `check_order_status`."),
rebuild, then test:

```python
print(assistant.chat("Where is my order 1234?", thread_id="order-test"))
```

Each new ability is just another `@tool` — the agent loop and everything else stays the same.

</details>

**Exercise 3 — Isolated per-customer memory.** Give two customers separate `thread_id`s. Tell
one their name, then ask the other thread for it — confirm memory stays isolated per thread.

*Sample run (illustrative):*

```text
Nice to meet you, Bilal! How can I help today?
I'm sorry, I don't have your name on record. Could you tell me?
Your name is Bilal.
```

<details><summary>Show solution</summary>

```python
print(assistant.chat("My name is Bilal.", thread_id="cust-A"))
print(assistant.chat("What's my name?", thread_id="cust-B"))   # should NOT know
print(assistant.chat("What's my name?", thread_id="cust-A"))   # should know
```

The `thread_id` keys the checkpointer, so one customer's context never bleeds into another's.

</details>

**Exercise 4 — Escalate urgent tickets.** After running the section-3 conversation (which files
a ticket), filter `assistant.tickets` for `priority == "high"` to build an escalation list — the
kind of routing a production helpdesk needs.

*Sample run (illustrative):*

```text
1 high-priority ticket(s) to escalate:
  - [software] Pixel Pro shuts off randomly after update
```

<details><summary>Show solution</summary>

```python
urgent = [t for t in assistant.tickets if t.priority == "high"]
print(f"{len(urgent)} high-priority ticket(s) to escalate:")
for t in urgent:
    print(f"  - [{t.category}] {t.subject}")
```

Because `priority` is a validated `Literal`, this filter is safe — no stray "High!"/"urgent"
values to handle. That's structured output paying off downstream.

</details>

**Exercise 5 — Smoke-test the assistant.** Write a tiny evaluation: a list of
`(question, expected_substring)` pairs, run each through `assistant.chat`, and flag any answer
that's missing the expected fact — a regression check to run whenever you change the prompt or
model.

*Sample run (illustrative):*

```text
[PASS] How long is the warranty?
        -> The Pixel comes with a 24-month limited warranty covering manufactu...
[PASS] What languages does Pixel Pro support?
        -> Pixel Pro supports English, Spanish, French, German, and Japanese...
```

<details><summary>Show solution</summary>

```python
checks = [
    ("How long is the warranty?", "24"),     # expect the 24-month figure
    ("What languages does Pixel Pro support?", "English"),
]
for q, expected in checks:
    ans = assistant.chat(q, thread_id="eval")
    ok = expected.lower() in ans.lower()
    print(f"[{'PASS' if ok else 'FAIL'}] {q}\n        -> {ans[:70]}...")
```

A handful of checks like these catch the moment a prompt tweak or model swap quietly breaks an
answer — cheap insurance for a production assistant.

</details>

## Course wrap-up

You've gone from a single `model.invoke("hello")` to a complete, memory-equipped, tool-using
assistant grounded in your own documents. The throughline of the whole course:

- **Standard interfaces** let you swap models, embeddings, and vector stores freely.
- **Composition** (LCEL `|`) builds complex behavior from small, testable pieces.
- **Grounding** (RAG) keeps answers accurate and citable.
- **Agency** (tools + `create_agent`) lets the model act, not just talk.
- **Structure and memory** make the whole thing reliable and conversational.

From here, the official docs at <https://docs.langchain.com> and the LangGraph guides are
your next stop — especially for advanced agent design, streaming, evaluation, and
deployment. You now have the mental model to read them with confidence.

Congratulations on finishing the course.